# Week 6 · Class 2 — Feature Engineering, Pipelines & Regularisation 🛠️🧪

**90 min · hands-on** · tools: `pandas`, `matplotlib`, `scikit-learn`

Tuesday we built a first linear regression on California housing. Today we make it **better** (features), **safer** (pipelines, no leakage) and **simpler** (regularisation).

| Part | Time | What we do |
|---|---|---|
| 0 | 3' | Setup: the full housing data (with missing values and a text category) |
| 1 | 5' | Tuesday's baseline: the score to beat |
| 2 | 15' | Numerical features: ratios, logs, scaling, bins |
| 3 | 8' | Categorical features: one-hot encoding |
| 4 | 4' | Dates and text (a tiny toy table) |
| 5 | 7' | Missing values: impute and flag |
| 6 | 10' | Data leakage: two ways to fool yourself |
| 7 | 13' | Pipeline + ColumnTransformer: the full recipe in one object |
| 8 | 10' | Overfitting: polynomial features |
| 9 | 12' | Regularisation: Ridge and Lasso |
| 10 | 8' | Choosing alpha with cross-validation |
| 11 | 3' | The results ladder |

🧑‍💻 = we write it **together**. 📚 **At home:** Part B at the end (5 exercises with solutions).

---
## 0 · Setup ⚙️

Today we use the **full** version of the California housing data: 20,640 districts, the same columns as Tuesday **plus**:
- `ocean_proximity`: a **text category** (how close to the ocean)
- some **missing values** in `total_bedrooms`

It loads straight from the internet, in Colab or locally.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder, PolynomialFeatures, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.metrics import r2_score, mean_absolute_error

pd.set_option("display.float_format", "{:,.3f}".format)

URL = "https://raw.githubusercontent.com/ageron/handson-ml2/master/datasets/housing/housing.csv"
housing = pd.read_csv(URL)
print(housing.shape)
housing.head()

🧑‍💻 **Code with me:** what is new compared with Tuesday?

In [ ]:
print(housing.isna().sum(), "\n")
print(housing["ocean_proximity"].value_counts())

📖 Two problems a model cannot handle directly: **207 missing values** and a **text column**. Today we solve both.

First, the split. We do it **now, before anything else**, and never look at the test set until the end.

In [ ]:
X = housing.drop(columns="median_house_value")
y = housing["median_house_value"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

results = {}          # we'll collect every model's scores here
def score(name, model, Xtr=None, Xte=None):
    # fit on train, store train and test R²
    Xtr = X_train if Xtr is None else Xtr
    Xte = X_test if Xte is None else Xte
    model.fit(Xtr, y_train)
    tr, te = r2_score(y_train, model.predict(Xtr)), r2_score(y_test, model.predict(Xte))
    results[name] = (tr, te)
    print(f"{name:<34} train R² = {tr:6.3f}   test R² = {te:6.3f}")
    return model

print(X_train.shape, X_test.shape)

---
## 1 · Tuesday's baseline 🪨

Tuesday's recipe: numeric columns only, linear regression. With missing values the model **crashes**, so the quick-and-dirty fix is to drop the text column and fill NaN with 0 (a bad idea we'll fix later).

In [ ]:
num_cols = X.select_dtypes("number").columns.tolist()
_ = score("baseline: raw numeric", LinearRegression(),
          X_train[num_cols].fillna(0), X_test[num_cols].fillna(0))

This is the number to beat today. 🎯

---
## 2 · Numerical features 🔢

### 2.1 Ratios: remove the size effect
`total_rooms` mostly tells us how **big** a district is. Per household, it tells us what the **homes** are like.

🧑‍💻 **Code with me:** a function that adds three ratio features.

In [ ]:
def add_ratios(df):
    df = df.copy()
    df["rooms_per_household"] = df["total_rooms"] / df["households"]
    df["bedrooms_per_room"] = df["total_bedrooms"] / df["total_rooms"]
    df["people_per_household"] = df["population"] / df["households"]
    return df

tmp = add_ratios(housing)
tmp[["total_rooms", "rooms_per_household", "total_bedrooms", "bedrooms_per_room", "median_house_value"]] \
    .corr()["median_house_value"]

📖 `bedrooms_per_room` is much more related to price than `total_bedrooms`: same information, better question.

### 2.2 Logs: tame long right tails
Counts like `population` have a few **huge** districts. `np.log1p(x)` = log(1 + x) squeezes the tail (and works with 0).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].hist(housing["population"], bins=60); axes[0].set_title("population")
axes[1].hist(np.log1p(housing["population"]), bins=60, color="green"); axes[1].set_title("log(1 + population)")
plt.show()

### 2.3 Scaling: the same ruler for every feature
- `StandardScaler`: mean 0, std 1
- `MinMaxScaler`: squeezed into [0, 1]

⚠️ Like every transformer: **`.fit` on train only**, then `.transform` both.

🧑‍💻 **Code with me:**

In [ ]:
cols = ["median_income", "housing_median_age", "total_rooms"]
scaler = StandardScaler().fit(X_train[cols])          # learns mean and std from TRAIN
scaled = pd.DataFrame(scaler.transform(X_train[cols]), columns=cols)
print("raw:\n", X_train[cols].describe().loc[["mean", "std"]])
print("\nstandardised:\n", scaled.describe().loc[["mean", "std"]])

In [ ]:
mm = MinMaxScaler().fit(X_train[cols])
pd.DataFrame(mm.transform(X_train[cols]), columns=cols).describe().loc[["min", "max"]]

### 2.4 Bins: from a number to a category
Sometimes the effect is step-like. `pd.cut` turns income into groups.

In [ ]:
income_cat = pd.cut(housing["median_income"], bins=[0, 2, 4, 6, np.inf], labels=["low", "mid", "high", "very high"])
housing.groupby(income_cat, observed=True)["median_house_value"].mean()

---
## 3 · Categorical features 🏷️

A model cannot multiply `"INLAND"` by a number. **One-hot encoding**: one 0/1 column per category.

🧑‍💻 **Code with me:** the quick pandas way first.

In [ ]:
pd.get_dummies(housing["ocean_proximity"]).head()

For ML we use scikit-learn's `OneHotEncoder`, because it **remembers the categories from train**. Two useful options:
- `handle_unknown="ignore"`: a category never seen in training becomes all zeros instead of an error
- `min_frequency=50`: rare categories (look at `ISLAND`: 5 districts!) are grouped together

In [ ]:
ohe = OneHotEncoder(handle_unknown="ignore", min_frequency=50, sparse_output=False)
ohe.fit(X_train[["ocean_proximity"]])
print(ohe.get_feature_names_out())
ohe.transform(pd.DataFrame({"ocean_proximity": ["INLAND", "ISLAND", "MOON BASE"]}))

📖 `ISLAND` went into the "infrequent" column, and `"MOON BASE"` (unknown) became all zeros: no crash in production.

> ⚠️ Never encode categories as 1, 2, 3… unless there is a **real order** (S < M < L → `OrdinalEncoder`). Otherwise the model thinks NEAR BAY = 3 × INLAND.

---
## 4 · Dates and text: the basics 📅📝

Our housing data has neither, so here is a tiny table of apartment listings.

In [ ]:
listings = pd.DataFrame({
    "listed_on": ["2026-11-02", "2026-11-07", "2026-11-14", "2026-11-15", "2026-12-01", "2026-12-24"],
    "description": ["Sunny flat close to metro", "Quiet studio", "Large family home with garden near metro",
                    "Renovated loft", "Cosy room, metro 5 min", "Penthouse with terrace and sea view"],
    "price": [1250, 780, 2400, 1600, 690, 3900],
})
listings

🧑‍💻 **Code with me:** turn the date and the text into numbers.

In [ ]:
listings["listed_on"] = pd.to_datetime(listings["listed_on"])
listings["month"] = listings["listed_on"].dt.month
listings["weekday"] = listings["listed_on"].dt.dayofweek          # 0 = Monday
listings["is_weekend"] = listings["weekday"] >= 5
listings["n_words"] = listings["description"].str.split().str.len()
listings["mentions_metro"] = listings["description"].str.contains("metro", case=False)
listings

---
## 5 · Missing values: impute and flag 🕳️

Instead of `fillna(0)` (which says "zero bedrooms"!):
- **impute** with the median of the **training** data
- **flag** the rows that were missing, in case missingness itself is a signal

🧑‍💻 **Code with me:**

In [ ]:
imp = SimpleImputer(strategy="median", add_indicator=True)
imp.fit(X_train[["total_bedrooms"]])
print("median learned from train:", imp.statistics_)
print(imp.get_feature_names_out())

rows_with_nan = X_train[X_train["total_bedrooms"].isna()].head(3)
imp.transform(rows_with_nan[["total_bedrooms"]])

---
## 6 · Data leakage: two ways to fool yourself 🕵️

**Leakage** = information the model would not have at prediction time sneaks into training. The result: a score that looks great and a model that fails in real life.

### 6.1 Target leakage: a feature built from the answer
Someone adds the district's yearly **property tax**. Sounds like useful information… but the tax was **computed from the sale price** (1.1% of the value).

🧑‍💻 **Code with me:**

In [ ]:
rng = np.random.default_rng(0)
leaky_train = X_train[num_cols].fillna(0).assign(property_tax=0.011 * y_train + rng.normal(0, 300, len(y_train)))
leaky_test = X_test[num_cols].fillna(0).assign(property_tax=0.011 * y_test + rng.normal(0, 300, len(y_test)))

m = LinearRegression().fit(leaky_train, y_train)
print("test R² with the leaky feature:", round(r2_score(y_test, m.predict(leaky_test)), 3))

😱 R² ≈ 0.95! Because `property_tax` **contains the price**. For a district we want to price, the sale has not happened yet, so this feature does not exist at prediction time.

> 🚨 **Too good to be true? Check for leakage first.**

### 6.2 Preprocessing leakage: fitting on all the data
Scaling or imputing **before** the split lets the test rows influence the means and medians. On 20,000 rows the effect is tiny, but on small data or time series it can be big. The cure is the same either way: **split first, then fit every step on train only**. Pipelines do this automatically.

In [ ]:
wrong_scaler = StandardScaler().fit(X[["median_income"]])          # ❌ saw the test rows
right_scaler = StandardScaler().fit(X_train[["median_income"]])    # ✅ train only
print("mean used (all data):", wrong_scaler.mean_, "  mean used (train):", right_scaler.mean_)

---
## 7 · Pipeline + ColumnTransformer: the full recipe 🧩

Different columns need different steps:
- **numeric**: add ratios → impute median → log → scale
- **categorical**: one-hot

`ColumnTransformer` routes each group of columns to its own steps; `Pipeline` chains everything with the model.

We split the columns into **three groups**:
- **counts** (rooms, people, income…): add ratios → impute median → log → scale
- **location** (longitude, latitude): only scale. ⚠️ No log here: longitude is negative and `log1p(-122)` does not exist.
- **category** (`ocean_proximity`): one-hot

🧑‍💻 **Code with me:** first a small helper so the pipeline knows the names of the 3 new ratio columns.

In [ ]:
def ratio_names(transformer, names):
    return list(names) + ["rooms_per_household", "bedrooms_per_room", "people_per_household"]

Now the three branches and the `ColumnTransformer`:

In [ ]:
geo_cols = ["longitude", "latitude"]
count_cols = ["housing_median_age", "total_rooms", "total_bedrooms", "population", "households", "median_income"]
cat_cols = ["ocean_proximity"]

count_pipe = make_pipeline(
    FunctionTransformer(add_ratios, feature_names_out=ratio_names),
    SimpleImputer(strategy="median"),
    FunctionTransformer(np.log1p, feature_names_out="one-to-one"),
    StandardScaler(),
)

prep = ColumnTransformer([
    ("counts", count_pipe, count_cols),
    ("geo", StandardScaler(), geo_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=50), cat_cols),
])
prep

🧑‍💻 **Code with me:** `add_ratios` needs `total_rooms`, `total_bedrooms`, `population` and `households`: all are in `count_cols`. Let's look at what comes out.

In [ ]:
prep.fit(X_train)
print(prep.transform(X_train).shape)
prep.get_feature_names_out()

Now the full pipeline with a linear regression at the end:

In [ ]:
pipe_lin = make_pipeline(prep, LinearRegression())
_ = score("pipeline: features + one-hot", pipe_lin)

📖 One `.fit`, one `.predict`, raw data in, prediction out. All the medians, means and categories were learned on **train only**: no leakage by construction. And a better score than the baseline, from features alone. 🎉

---
## 8 · Overfitting: when more features hurt 📈📉

`PolynomialFeatures` adds squares and products of features (x², x₁·x₂…), so a linear model can draw curves.

🧑‍💻 **Code with me:** degree 2, then degree 3. Watch the gap between train and test.

In [ ]:
pipe_p2 = make_pipeline(prep, PolynomialFeatures(2, include_bias=False), LinearRegression())
_ = score("polynomial degree 2", pipe_p2)

pipe_p3 = make_pipeline(prep, PolynomialFeatures(3, include_bias=False), LinearRegression())
_ = score("polynomial degree 3, no penalty", pipe_p3)

In [ ]:
n_features = pipe_p3[:-1].transform(X_train.head()).shape[1]
print("features after degree 3:", n_features)

📖 Degree 3 has a high train R², but the test score is clearly worse: with so many features the model starts fitting **noise** (it can even go negative on some splits). This is **overfitting**: big train–test gap.

---
## 9 · Regularisation: Ridge and Lasso ⚖️

Add a penalty for big coefficients:
- **Ridge (L2)**: errors + α · Σ b²  → shrinks all coefficients
- **Lasso (L1)**: errors + α · Σ |b|  → sets some coefficients to **exactly 0**

α = 0 → plain linear regression; α large → everything pushed towards 0. Our pipeline already **scales**, which the penalty needs.

🧑‍💻 **Code with me:** the degree-3 model, now with Ridge.

In [ ]:
pipe_p3_ridge = make_pipeline(prep, PolynomialFeatures(3, include_bias=False), Ridge(alpha=300))
_ = score("degree 3 + Ridge (α=300)", pipe_p3_ridge)

📖 The penalty pulls the test score back up: same features, but the model is no longer allowed to use crazy coefficients.

🧑‍💻 **Code with me:** how does α change the coefficients? (No polynomials here, so we can read them.)

In [ ]:
alphas = np.logspace(-2, 5, 40)
coefs = []
for a in alphas:
    m = make_pipeline(prep, Ridge(alpha=a)).fit(X_train, y_train)
    coefs.append(m[-1].coef_)

plt.figure(figsize=(8, 4))
plt.semilogx(alphas, np.array(coefs) / 1000)
plt.axhline(0, color="black", lw=0.8)
plt.xlabel("alpha"); plt.ylabel("coefficient (k$)"); plt.title("Ridge: coefficients shrink as alpha grows")
plt.show()

🧑‍💻 **Code with me:** Lasso as a feature selector.

In [ ]:
names = prep.get_feature_names_out()
for a in [10, 1000, 5000]:
    lasso = make_pipeline(prep, Lasso(alpha=a, max_iter=20_000)).fit(X_train, y_train)
    kept = names[np.abs(lasso[-1].coef_) > 1e-6]
    print(f"alpha={a:>5}: keeps {len(kept):>2} of {len(names)} features")
print("\nstill kept at alpha=5000:", list(kept))

---
## 10 · Choosing alpha with cross-validation 🔁

We must **not** choose α by looking at the test set (that would be leakage too). Instead:
- split the **training** data into 5 folds
- train on 4, validate on 1, rotate, average
- keep the α with the best average, then test **once**

🧑‍💻 **Code with me:** first by hand with `cross_val_score`.

In [ ]:
for a in [0.1, 1, 10, 100, 1000]:
    pipe = make_pipeline(prep, PolynomialFeatures(2, include_bias=False), Ridge(alpha=a))
    cv = cross_val_score(pipe, X_train, y_train, cv=5, scoring="r2")
    print(f"alpha={a:>6}: mean CV R² = {cv.mean():.3f}  (± {cv.std():.3f})")

Now let `GridSearchCV` try **degree and α together**. Step names come from `make_pipeline` (lower-case class names).

In [ ]:
pipe = make_pipeline(prep, PolynomialFeatures(include_bias=False), Ridge())
grid = {"polynomialfeatures__degree": [1, 2, 3],
        "ridge__alpha": [0.1, 1, 10, 100, 1000]}
search = GridSearchCV(pipe, grid, cv=5, scoring="r2", n_jobs=-1)
search.fit(X_train, y_train)
print("best params:", search.best_params_)
print("best mean CV R²:", round(search.best_score_, 3))

And only now, **once**, the test set:

In [ ]:
best = search.best_estimator_
results["best by CV (GridSearch)"] = (r2_score(y_train, best.predict(X_train)), r2_score(y_test, best.predict(X_test)))
print("test R²:", round(results["best by CV (GridSearch)"][1], 3))
print("test MAE: $", f"{mean_absolute_error(y_test, best.predict(X_test)):,.0f}")

---
## 11 · The results ladder 🪜

In [ ]:
res = pd.DataFrame(results, index=["train R²", "test R²"]).T
display(res)

ax = res["test R²"].clip(lower=0).plot(kind="barh", figsize=(8, 3.5), color="tab:blue")
ax.invert_yaxis(); ax.set_xlabel("test R² (negative shown as 0)"); ax.set_xlim(0, 1)
plt.show()

🧭 **What we learned**
- Better features and a proper pipeline beat the raw baseline, with the **same** linear model.
- More features can **overfit**: watch the train–test gap.
- **Regularisation** lets us keep the flexibility without the collapse.
- Choose α (and other settings) with **cross-validation**, test **once**.

> 🤔 *"Degree 3 + Ridge (α=300)" scores a hair higher on test than the CV winner. Should we pick it?* **No.** That α was not chosen by CV, and choosing a model because of its test score turns the test set into a validation set: leakage again. The CV winner's score is the honest one.
- Week 8: trees and boosting go even further.

---
# 📚 Part B · At home — 5 exercises

Try first, then compare with the solution cell.

### Exercise 1 · Elastic Net 🧪
`ElasticNet` mixes Ridge and Lasso through `l1_ratio` (0 = Ridge, 1 = Lasso). Use `GridSearchCV` on `make_pipeline(prep, ElasticNet(max_iter=20_000))` with `alpha` in [1, 10, 100] and `l1_ratio` in [0.1, 0.5, 0.9]. Best params and test R²?

In [ ]:
from sklearn.linear_model import ElasticNet
en = GridSearchCV(make_pipeline(prep, ElasticNet(max_iter=20_000)),
                  {"elasticnet__alpha": [1, 10, 100], "elasticnet__l1_ratio": [0.1, 0.5, 0.9]},
                  cv=5, scoring="r2", n_jobs=-1).fit(X_train, y_train)
print(en.best_params_, "  test R²:", round(r2_score(y_test, en.predict(X_test)), 3))

### Exercise 2 · Your own feature 💡
Add a feature `income_per_person = median_income * households / population` inside `add_ratios` (copy it as `add_ratios_v2`), rebuild the pipeline and compare test R² with `pipe_lin`.

*Hint:* you also need a new `ratio_names` function with the extra name.

In [ ]:
def add_ratios_v2(df):
    df = add_ratios(df)
    df["income_per_person"] = df["median_income"] * df["households"] / df["population"]
    return df
def ratio_names_v2(transformer, names):
    return ratio_names(transformer, names) + ["income_per_person"]

count_pipe_v2 = make_pipeline(FunctionTransformer(add_ratios_v2, feature_names_out=ratio_names_v2),
                              SimpleImputer(strategy="median"),
                              FunctionTransformer(np.log1p, feature_names_out="one-to-one"), StandardScaler())
prep_v2 = ColumnTransformer([("counts", count_pipe_v2, count_cols), ("geo", StandardScaler(), geo_cols),
                             ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=50), cat_cols)])
_ = score("pipeline + income_per_person", make_pipeline(prep_v2, LinearRegression()))

### Exercise 3 · Random search 🎲
A grid gets slow when you have many settings. `RandomizedSearchCV` tries a fixed number of random combinations. Search `ridge__alpha` from `np.logspace(-2, 4, 100)` and degree in [1, 2] with `n_iter=15`.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
rs = RandomizedSearchCV(make_pipeline(prep, PolynomialFeatures(include_bias=False), Ridge()),
                        {"polynomialfeatures__degree": [1, 2], "ridge__alpha": np.logspace(-2, 4, 100)},
                        n_iter=15, cv=5, scoring="r2", random_state=0, n_jobs=-1).fit(X_train, y_train)
print(rs.best_params_, "  CV R²:", round(rs.best_score_, 3))

### Exercise 4 · Find the leak 🕵️
A colleague sends this dataset and is proud of the R². Which column is the problem, and why?

In [ ]:
rng = np.random.default_rng(1)
colleague = X_train[num_cols].fillna(0).copy()
colleague["days_on_market"] = rng.integers(5, 120, len(colleague))
colleague["agent_commission"] = 0.03 * y_train + rng.normal(0, 500, len(colleague))   # paid AFTER the sale
colleague.head()

In [ ]:
Xa, Xb, ya, yb = train_test_split(colleague, y_train, test_size=0.2, random_state=0)
for cols in [num_cols, num_cols + ["days_on_market"], num_cols + ["agent_commission"]]:
    m = LinearRegression().fit(Xa[cols], ya)
    print(f"{cols[-1]:<20} R² = {r2_score(yb, m.predict(Xb[cols])):.3f}")

`agent_commission` is 3% of the sale price, known only **after** the sale: target leakage. R² ≈ 1 is the warning sign.

### Exercise 5 · Save and reuse the model 💾
Save `search.best_estimator_` with `joblib`, load it back, and predict the value of the first test district from its **raw** columns.

In [ ]:
import joblib
joblib.dump(search.best_estimator_, "california_model.joblib")
loaded = joblib.load("california_model.joblib")
one = X_test.head(1)
print(one.T)
print(f"\npredicted: ${loaded.predict(one)[0]:,.0f}   actual: ${y_test.iloc[0]:,.0f}")

The whole recipe (ratios, imputer, log, scaler, one-hot, polynomials, Ridge) travels in **one file**. That is what gets deployed. 🚀